# Explore Hybrid HDBSCAN Clustering (KNN + Closest Face)

Interactive notebook for exploring `HybridHDBSCANKNN` and `HybridHDBSCANClosestFace` parameters and visualizing results.

**Features:**
- Switch between two hybrid HDBSCAN algorithms with one variable
- Parameter sweep with configurable param grids for each algorithm
- UMAP visualization colored by cluster
- Display top N faces from K largest clusters
- Inter-cluster distance analysis


In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Any, Tuple
from itertools import product
import json
from IPython.display import display, HTML
import base64
from PIL import Image
import io

# UMAP for visualization
try:
    import umap
    HAS_UMAP = True
except ImportError:
    print("UMAP not installed. Run: pip install umap-learn")
    HAS_UMAP = False

# Local imports
from sim_bench.clustering.base import load_clustering_method
from sim_bench.clustering.distance_utils import (
    cosine_distance_matrix,
    closest_distance_to_cluster,
    all_cluster_distances,
    compute_cluster_members_from_labels,
)

print("Imports complete")

## 1. Load Data

In [ ]:
# Configuration
RESULTS_DIR = Path("../results/face_clustering_benchmark")

# Load embeddings (most recent)
npy_files = sorted(RESULTS_DIR.glob("embeddings_*.npy"), reverse=True)
EMBEDDINGS_FILE = npy_files[0]
embeddings = np.load(EMBEDDINGS_FILE)

# Load metadata
json_files = sorted(RESULTS_DIR.glob("benchmark_*.json"), reverse=True)
with open(json_files[0]) as f:
    benchmark_data = json.load(f)
metadata = benchmark_data.get("face_metadata", [])

# Normalize embeddings
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / np.maximum(norms, 1e-10)

# Compute distance matrix once
distance_matrix = cosine_distance_matrix(embeddings_norm)

print(f"Loaded {len(embeddings)} embeddings from {EMBEDDINGS_FILE.name}")
print(f"Distance matrix shape: {distance_matrix.shape}")
print(f"Face crops dir: {RESULTS_DIR / 'face_crops'}")

## 2. Utility Functions

In [ ]:
def compute_umap(embeddings: np.ndarray, n_neighbors: int = 15, min_dist: float = 0.1) -> np.ndarray:
    """Compute UMAP coordinates for embeddings."""
    if not HAS_UMAP:
        raise ImportError("UMAP not available")
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=2, random_state=42)
    return reducer.fit_transform(embeddings)


def get_face_image(face_idx: int, results_dir: Path = RESULTS_DIR) -> Image.Image:
    """Load face crop image."""
    crops_dir = results_dir / "face_crops"
    for pattern in [f"face_{face_idx:04d}_aligned.jpg", f"face_{face_idx:04d}.jpg"]:
        path = crops_dir / pattern
        if path.exists():
            return Image.open(path)
    return None


def display_faces_grid(face_indices: List[int], cols: int = 10, size: int = 60):
    """Display faces in a grid using HTML."""
    html = '<div style="display:flex;flex-wrap:wrap;gap:2px;">'
    for idx in face_indices:
        img = get_face_image(idx)
        if img:
            img = img.resize((size, size))
            buffer = io.BytesIO()
            img.save(buffer, format='JPEG')
            b64 = base64.b64encode(buffer.getvalue()).decode()
            html += f'<img src="data:image/jpeg;base64,{b64}" title="#{idx}" style="border-radius:3px;">'
        else:
            html += f'<div style="width:{size}px;height:{size}px;background:#333;display:flex;align-items:center;justify-content:center;color:#888;font-size:9px;">#{idx}</div>'
    html += '</div>'
    display(HTML(html))


def _fmt_param_value(value: Any) -> str:
    if isinstance(value, float):
        return f"{value:.3g}"
    return str(value)


def format_params_short(params: Dict[str, Any], max_items: int = 5) -> str:
    """Compact parameter string for titles/logs."""
    aliases = {
        'cluster_selection_epsilon': 'eps',
        'threshold_percentile': 'pct',
        'threshold_floor': 'floor',
        'threshold_ceiling': 'ceil',
        'merge_min_pairs': 'merge_pairs',
        'merge_min_distinct': 'merge_dist',
        'attach_min_exemplars': 'attach_ex',
        'merge_min_faces': 'merge_faces',
        'merge_threshold_multiplier': 'merge_x',
        'early_exit_multiplier': 'early_x',
        'attach_min_neighbors': 'attach_n',
        'min_cluster_size': 'min_cl',
        'min_samples': 'min_s',
        'split_threshold': 'split_t',
        'split_k': 'split_k',
        'knn_k': 'knn_k',
    }
    items = []
    for key in sorted(params.keys()):
        label = aliases.get(key, key)
        items.append(f"{label}={_fmt_param_value(params[key])}")
    if len(items) > max_items:
        return ', '.join(items[:max_items]) + f", ... (+{len(items)-max_items})"
    return ', '.join(items)


def run_clustering(params: Dict[str, Any], embeddings: np.ndarray) -> Tuple[np.ndarray, Dict]:
    """Run clustering with given parameters for selected algorithm."""
    config = {
        'algorithm': ALGORITHM,
        'params': params,
    }
    clusterer = load_clustering_method(config)
    return clusterer.cluster(embeddings, collect_debug_data=True)


def cluster_summary(labels: np.ndarray, stats: Dict) -> Dict:
    """Generate cluster summary."""
    sizes = sorted(stats.get('cluster_sizes', {}).values(), reverse=True)
    return {
        'n_clusters': stats.get('n_clusters', 0),
        'n_noise': stats.get('n_noise', 0),
        'n_singletons': sum(1 for s in sizes if s == 1),
        'max_size': max(sizes) if sizes else 0,
        'top5_sizes': sizes[:5],
    }

print("Utility functions defined")


## 3. Choose Algorithm + Parameter Sweep


In [ ]:
# Choose algorithm to explore
ALGORITHM = 'hybrid_hdbscan_knn'  # or: 'hybrid_closest_face'

ALGORITHM_LABELS = {
    'hybrid_hdbscan_knn': 'HybridHDBSCANKNN',
    'hybrid_closest_face': 'HybridHDBSCANClosestFace',
}

# Define parameter grids as list of dicts (complete parameter sets)
# Start with these presets, then edit/expand as needed.
PARAM_GRIDS_BY_ALGORITHM = {
    'hybrid_hdbscan_knn': [
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 85, 'merge_min_pairs': 3, 'merge_min_distinct': 2,
         'attach_min_exemplars': 2, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 90, 'merge_min_pairs': 3, 'merge_min_distinct': 2,
         'attach_min_exemplars': 2, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.060, 'knn_k': 3,
         'threshold_percentile': 90, 'merge_min_pairs': 3, 'merge_min_distinct': 2,
         'attach_min_exemplars': 2, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 95, 'merge_min_pairs': 3, 'merge_min_distinct': 2,
         'attach_min_exemplars': 2, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 90, 'merge_min_pairs': 2, 'merge_min_distinct': 2,
         'attach_min_exemplars': 1, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 4,
         'threshold_percentile': 90, 'merge_min_pairs': 3, 'merge_min_distinct': 2,
         'attach_min_exemplars': 2, 'split_enabled': True, 'split_threshold': 0.70},
    ],
    'hybrid_closest_face': [
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 85, 'merge_min_faces': 2, 'merge_threshold_multiplier': 1.25,
         'early_exit_multiplier': 2.0, 'attach_min_neighbors': 1, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 90, 'merge_min_faces': 2, 'merge_threshold_multiplier': 1.50,
         'early_exit_multiplier': 2.0, 'attach_min_neighbors': 1, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.060, 'knn_k': 3,
         'threshold_percentile': 90, 'merge_min_faces': 2, 'merge_threshold_multiplier': 1.50,
         'early_exit_multiplier': 2.0, 'attach_min_neighbors': 1, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 95, 'merge_min_faces': 2, 'merge_threshold_multiplier': 1.50,
         'early_exit_multiplier': 2.5, 'attach_min_neighbors': 1, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 3,
         'threshold_percentile': 90, 'merge_min_faces': 3, 'merge_threshold_multiplier': 1.50,
         'early_exit_multiplier': 2.0, 'attach_min_neighbors': 1, 'split_enabled': True, 'split_threshold': 0.65},
        {'min_cluster_size': 2, 'min_samples': 2, 'cluster_selection_epsilon': 0.045, 'knn_k': 4,
         'threshold_percentile': 90, 'merge_min_faces': 2, 'merge_threshold_multiplier': 1.75,
         'early_exit_multiplier': 2.0, 'attach_min_neighbors': 1, 'split_enabled': True, 'split_threshold': 0.70},
    ],
}

if ALGORITHM not in PARAM_GRIDS_BY_ALGORITHM:
    raise ValueError(f"Unknown ALGORITHM={ALGORITHM!r}. Options: {sorted(PARAM_GRIDS_BY_ALGORITHM)}")

PARAM_GRID = PARAM_GRIDS_BY_ALGORITHM[ALGORITHM]

print(f"Algorithm: {ALGORITHM_LABELS.get(ALGORITHM, ALGORITHM)} ({ALGORITHM})")
print(f"Parameter grid: {len(PARAM_GRID)} configurations")


In [ ]:
# Run parameter sweep
results = []

for i, params in enumerate(PARAM_GRID):
    labels, stats = run_clustering(params, embeddings_norm)
    summary = cluster_summary(labels, stats)

    results.append({
        'algorithm': ALGORITHM,
        'params': params,
        'labels': labels,
        'stats': stats,
        'summary': summary,
    })

    print(
        f"[{i+1}/{len(PARAM_GRID)}] {format_params_short(params, max_items=4)}: "
        f"{summary['n_clusters']} clusters, {summary['n_noise']} noise, max={summary['max_size']}, "
        f"sizes={summary['top5_sizes']}"
    )

print(f"\nCompleted {len(results)} configurations for {ALGORITHM}")


In [ ]:
# Display results table
import pandas as pd

rows = []
for idx, r in enumerate(results):
    p = r['params']
    s = r['summary']
    row = {
        'idx': idx,
        'algorithm': r.get('algorithm', ALGORITHM),
        'clusters': s['n_clusters'],
        'noise': s['n_noise'],
        'singletons': s['n_singletons'],
        'max_size': s['max_size'],
        'top5': str(s['top5_sizes']),
    }
    row.update(p)
    rows.append(row)

df = pd.DataFrame(rows)
summary_cols = ['idx', 'algorithm', 'clusters', 'noise', 'singletons', 'max_size', 'top5']
param_cols = sorted([c for c in df.columns if c not in summary_cols])
df = df[summary_cols + param_cols]
display(df)


## 4. Select Configuration for Detailed Analysis


In [ ]:
# Select which result to analyze in detail
# Change this index to explore different configurations
SELECTED_IDX = 0  # Index into results list

if not results:
    raise ValueError("No results available. Run the parameter sweep cell first.")
if not (0 <= SELECTED_IDX < len(results)):
    raise IndexError(f"SELECTED_IDX={SELECTED_IDX} out of range for {len(results)} results")

selected = results[SELECTED_IDX]
selected_params = selected['params']
selected_labels = selected['labels']
selected_stats = selected['stats']

print(f"Selected configuration {SELECTED_IDX} ({selected.get('algorithm', ALGORITHM)}):")
print(f"  Parameters: {selected_params}")
print(f"  Summary: {selected['summary']}")


## 5. UMAP Visualization

In [ ]:
# Compute UMAP (cached for reuse)
if 'umap_coords' not in dir():
    print("Computing UMAP...")
    umap_coords = compute_umap(embeddings_norm)
    print(f"UMAP complete: shape {umap_coords.shape}")
else:
    print(f"Using cached UMAP coords: shape {umap_coords.shape}")

In [ ]:
def plot_umap_clusters(umap_coords: np.ndarray, labels: np.ndarray, title: str = "UMAP Clusters"):
    """Plot UMAP with cluster coloring."""
    fig, ax = plt.subplots(figsize=(12, 10))

    unique_labels = sorted(set(labels))
    colors = plt.cm.tab20(np.linspace(0, 1, max(20, len(unique_labels))))

    for i, label in enumerate(unique_labels):
        mask = labels == label
        if label == -1:
            ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                      c='gray', s=20, alpha=0.5, label=f'Noise ({mask.sum()})')
        else:
            ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                      c=[colors[i % len(colors)]], s=30, alpha=0.7,
                      label=f'C{label} ({mask.sum()})')

    ax.set_title(title, fontsize=14)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')

    # Legend outside plot
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()

# Plot selected configuration
title = (
    f"{ALGORITHM}: {format_params_short(selected_params, max_items=4)} | "
    f"{selected['summary']['n_clusters']} clusters"
)
plot_umap_clusters(umap_coords, selected_labels, title)


## 6. Display Top N Faces from K Largest Clusters

In [ ]:
# Configuration
K_CLUSTERS = 10  # Number of largest clusters to show
N_FACES = 10     # Number of faces per cluster

def display_top_clusters(labels: np.ndarray, k_clusters: int = 10, n_faces: int = 10):
    """Display top N faces from K largest clusters."""
    cluster_members = compute_cluster_members_from_labels(labels)
    
    # Sort clusters by size (exclude noise)
    clusters_by_size = sorted(
        [(cid, members) for cid, members in cluster_members.items() if cid != -1],
        key=lambda x: len(x[1]),
        reverse=True
    )
    
    print(f"Showing top {min(k_clusters, len(clusters_by_size))} clusters (of {len(clusters_by_size)} total)\n")
    
    for i, (cluster_id, members) in enumerate(clusters_by_size[:k_clusters]):
        print(f"\n{'='*60}")
        print(f"Cluster {cluster_id} ({len(members)} faces) - showing first {min(n_faces, len(members))}")
        print(f"{'='*60}")
        display_faces_grid(members[:n_faces])

display_top_clusters(selected_labels, K_CLUSTERS, N_FACES)

## 7. Inter-Cluster Distance Analysis

In [ ]:
def compute_inter_cluster_distances(labels: np.ndarray, distance_matrix: np.ndarray) -> Dict:
    """Compute min/mean distances between all pairs of clusters."""
    cluster_members = compute_cluster_members_from_labels(labels)
    cluster_ids = sorted([c for c in cluster_members.keys() if c != -1])
    
    n_clusters = len(cluster_ids)
    min_dists = np.zeros((n_clusters, n_clusters))
    mean_dists = np.zeros((n_clusters, n_clusters))
    
    for i, ci in enumerate(cluster_ids):
        for j, cj in enumerate(cluster_ids):
            if i == j:
                # Intra-cluster distances
                members = cluster_members[ci]
                if len(members) > 1:
                    dists = distance_matrix[np.ix_(members, members)]
                    np.fill_diagonal(dists, np.inf)
                    min_dists[i, j] = np.min(dists)
                    mean_dists[i, j] = np.mean(dists[dists < np.inf])
            else:
                # Inter-cluster distances
                mi, mj = cluster_members[ci], cluster_members[cj]
                dists = distance_matrix[np.ix_(mi, mj)]
                min_dists[i, j] = np.min(dists)
                mean_dists[i, j] = np.mean(dists)
    
    return {
        'cluster_ids': cluster_ids,
        'min_distances': min_dists,
        'mean_distances': mean_dists,
    }

# Compute for selected configuration
inter_cluster = compute_inter_cluster_distances(selected_labels, distance_matrix)
print(f"Computed distances between {len(inter_cluster['cluster_ids'])} clusters")

In [ ]:
def plot_inter_cluster_heatmap(inter_cluster: Dict, title: str = "Inter-Cluster Distances"):
    """Plot heatmap of inter-cluster distances."""
    cluster_ids = inter_cluster['cluster_ids']
    min_dists = inter_cluster['min_distances']
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(min_dists, cmap='viridis')
    
    ax.set_xticks(range(len(cluster_ids)))
    ax.set_yticks(range(len(cluster_ids)))
    ax.set_xticklabels([f'C{c}' for c in cluster_ids], fontsize=8)
    ax.set_yticklabels([f'C{c}' for c in cluster_ids], fontsize=8)
    
    plt.colorbar(im, label='Min Distance')
    ax.set_title(f"{title}\n(Min distance between any two faces)")
    plt.tight_layout()
    plt.show()

plot_inter_cluster_heatmap(
    inter_cluster,
    f"Inter-Cluster Distances ({ALGORITHM}, {format_params_short(selected_params, max_items=3)})"
)


In [ ]:
def find_closest_cluster_pairs(inter_cluster: Dict, top_n: int = 10):
    """Find the N closest pairs of different clusters."""
    cluster_ids = inter_cluster['cluster_ids']
    min_dists = inter_cluster['min_distances']
    
    pairs = []
    n = len(cluster_ids)
    for i in range(n):
        for j in range(i+1, n):
            pairs.append({
                'cluster_a': cluster_ids[i],
                'cluster_b': cluster_ids[j],
                'min_dist': min_dists[i, j],
            })
    
    pairs.sort(key=lambda x: x['min_dist'])
    return pairs[:top_n]

# Find closest cluster pairs
closest_pairs = find_closest_cluster_pairs(inter_cluster, top_n=10)
print("Top 10 closest cluster pairs (potential merge candidates):\n")
for p in closest_pairs:
    print(f"  C{p['cluster_a']} <-> C{p['cluster_b']}: min_dist = {p['min_dist']:.4f}")

## 8. Detailed Distance Analysis: Specific Clusters

In [ ]:
def analyze_cluster_pair(cluster_a: int, cluster_b: int, labels: np.ndarray, 
                         distance_matrix: np.ndarray, n_show: int = 5):
    """Detailed analysis of distances between two clusters."""
    cluster_members = compute_cluster_members_from_labels(labels)
    
    members_a = cluster_members.get(cluster_a, [])
    members_b = cluster_members.get(cluster_b, [])
    
    if not members_a or not members_b:
        print(f"Empty cluster: A has {len(members_a)}, B has {len(members_b)} members")
        return
    
    # Compute cross-distances
    cross_dists = distance_matrix[np.ix_(members_a, members_b)]
    
    # Statistics
    print(f"\nCluster {cluster_a} ({len(members_a)} faces) vs Cluster {cluster_b} ({len(members_b)} faces)")
    print(f"Cross-cluster distances:")
    print(f"  Min: {np.min(cross_dists):.4f}")
    print(f"  Max: {np.max(cross_dists):.4f}")
    print(f"  Mean: {np.mean(cross_dists):.4f}")
    print(f"  Std: {np.std(cross_dists):.4f}")
    
    # Find closest pairs
    flat_idx = np.argsort(cross_dists.ravel())[:n_show]
    print(f"\nTop {n_show} closest pairs:")
    for idx in flat_idx:
        i, j = np.unravel_index(idx, cross_dists.shape)
        face_a, face_b = members_a[i], members_b[j]
        dist = cross_dists[i, j]
        print(f"  Face #{face_a} (C{cluster_a}) <-> Face #{face_b} (C{cluster_b}): dist={dist:.4f}")
    
    # Show faces from closest pair
    i, j = np.unravel_index(flat_idx[0], cross_dists.shape)
    face_a, face_b = members_a[i], members_b[j]
    
    print(f"\nClosest pair faces:")
    print(f"From Cluster {cluster_a}:")
    display_faces_grid([face_a], cols=1, size=100)
    print(f"From Cluster {cluster_b}:")
    display_faces_grid([face_b], cols=1, size=100)
    
    # Show all faces from both clusters
    print(f"\nAll faces in Cluster {cluster_a}:")
    display_faces_grid(members_a[:20])
    print(f"\nAll faces in Cluster {cluster_b}:")
    display_faces_grid(members_b[:20])

# Analyze the closest pair
if closest_pairs:
    pair = closest_pairs[0]
    analyze_cluster_pair(pair['cluster_a'], pair['cluster_b'], selected_labels, distance_matrix)

## 9. Custom Analysis: Distance from Image to Clusters

In [ ]:
def analyze_face_to_clusters(face_idx: int, labels: np.ndarray, distance_matrix: np.ndarray):
    """Analyze distances from a specific face to all clusters."""
    cluster_members = compute_cluster_members_from_labels(labels)
    current_cluster = int(labels[face_idx])
    
    print(f"\nFace #{face_idx} - Current cluster: {current_cluster}")
    display_faces_grid([face_idx], cols=1, size=100)
    
    # Get distances to all clusters
    dists_to_clusters = all_cluster_distances(face_idx, distance_matrix, cluster_members)
    
    # Sort by distance
    sorted_clusters = sorted(dists_to_clusters.items(), key=lambda x: x[1])
    
    print(f"\nDistances to all clusters (sorted by closest):")
    for cluster_id, dist in sorted_clusters:
        marker = " <-- current" if cluster_id == current_cluster else ""
        size = len(cluster_members.get(cluster_id, []))
        print(f"  C{cluster_id} (size={size}): {dist:.4f}{marker}")
    
    # Show closest faces from top 3 clusters
    print(f"\nClosest faces from top 3 clusters:")
    for cluster_id, dist in sorted_clusters[:3]:
        members = cluster_members[cluster_id]
        # Find closest face in this cluster
        dists_in_cluster = distance_matrix[face_idx, members]
        closest_idx = members[np.argmin(dists_in_cluster)]
        print(f"\nCluster {cluster_id} - closest face #{closest_idx} (dist={np.min(dists_in_cluster):.4f}):")
        display_faces_grid([closest_idx], cols=1, size=80)

# Example: analyze face 0
analyze_face_to_clusters(0, selected_labels, distance_matrix)

In [ ]:
# Interactive: change face_idx to explore different faces
FACE_TO_ANALYZE = 50  # Change this value
analyze_face_to_clusters(FACE_TO_ANALYZE, selected_labels, distance_matrix)

## 10. Compare Multiple Configurations Side-by-Side

In [ ]:
def compare_configs_umap(results_list: List[Dict], umap_coords: np.ndarray, max_plots: int = 4):
    """Plot UMAP for multiple configurations side by side."""
    n_plots = min(len(results_list), max_plots)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    colors = plt.cm.tab20(np.linspace(0, 1, 20))

    for plot_idx, (ax, result) in enumerate(zip(axes, results_list[:n_plots])):
        labels = result['labels']
        params = result['params']
        summary = result['summary']

        unique_labels = sorted(set(labels))

        for i, label in enumerate(unique_labels):
            mask = labels == label
            color = 'gray' if label == -1 else colors[i % len(colors)]
            ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                      c=[color], s=15, alpha=0.6)

        ax.set_title(
            f"#{plot_idx} {format_params_short(params, max_items=2)}\n"
            f"{summary['n_clusters']} clusters, {summary['n_noise']} noise",
            fontsize=10,
        )
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

# Compare first 4 configurations
compare_configs_umap(results, umap_coords, max_plots=4)


---

## Quick Reference

**Key Variables:**
- `ALGORITHM`: Select `'hybrid_hdbscan_knn'` or `'hybrid_closest_face'`
- `PARAM_GRIDS_BY_ALGORITHM`: Preset parameter sweeps for each algorithm
- `PARAM_GRID`: Active parameter grid for the selected algorithm
- `SELECTED_IDX`: Index of configuration to analyze in detail
- `K_CLUSTERS`: Number of largest clusters to display
- `N_FACES`: Number of faces to show per cluster

**Key Functions:**
- `run_clustering(params, embeddings)`: Run selected algorithm with params
- `format_params_short(params)`: Compact params string for logs/titles
- `display_faces_grid(indices)`: Show faces as image grid
- `analyze_cluster_pair(a, b, ...)`: Compare two clusters
- `analyze_face_to_clusters(idx, ...)`: Show distances from one face to all clusters
